# Tiny model: handwritten digits

A Colab-style notebook, and agent-testbench's first example. It trains a small neural network on scikit-learn's
bundled 8x8 digits (no download), saves its metrics and a confusion-matrix plot, and offers the metrics for
download. It runs as-is on Colab; `testbench run smoke` runs it headless on your machine, `testbench run full-modal`
on Modal.

The settings below are Colab form fields: a plan in `testbench.yaml` can set them (`params: {EPOCHS: 15}`).

In [ ]:
#@title Settings
HIDDEN = 64 #@param {type:"integer"}
EPOCHS = 60 #@param {type:"integer"}
LEARNING_RATE = 0.01 #@param {type:"number"}
SEED = 0 #@param {type:"integer"}

In [ ]:
#@title Data
from pathlib import Path
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
X, y = load_digits(return_X_y=True)
X = X / 16.0
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
print(f"{len(X_train)} training and {len(X_test)} test images of 8x8 digits")

In [ ]:
#@title Train
import json
import time
import warnings

from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore", category=ConvergenceWarning)
model = MLPClassifier(hidden_layer_sizes=(HIDDEN,), max_iter=EPOCHS, learning_rate_init=LEARNING_RATE, random_state=SEED)
began = time.time()
model.fit(X_train, y_train)
metrics = {
    "accuracy": round(float(model.score(X_test, y_test)), 4),
    "epochs": int(model.n_iter_),
    "final_loss": round(float(model.loss_), 4),
    "hidden": HIDDEN,
    "learning_rate": LEARNING_RATE,
    "seconds": round(time.time() - began, 2),
}
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics))

In [ ]:
#@title Where it goes wrong
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, ax=ax, colorbar=False)
ax.set_title(f"accuracy {metrics['accuracy']:.3f}")
fig.tight_layout()
fig.savefig(OUT / "confusion.png", dpi=120)
plt.show()

In [ ]:
#@title Download the metrics
from google.colab import files

files.download(str(OUT / "metrics.json"))